# 35. Structured Output: Getting Predictable Formats

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/05-output-control/35_structured_output.ipynb)

**Category:** Output Control & Formatting  **Technique #:** 35  **Difficulty:** Intermediate

## 📋 Description

Structured Output is a technique that guides LLMs to produce responses in a predictable, parseable format. By explicitly specifying the desired structure in your prompt, you can reliably extract data, integrate with APIs, and build robust applications.

**When to use:**
- When you need machine-readable responses
- Building data extraction pipelines
- Creating API integrations
- Generating reports with consistent formatting

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│  User Prompt + Structure Definition                        │
│  "Extract entities in this format: Name | Type | Value"   │
└─────────────────────────┬───────────────────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────┐
│  LLM Processes Request                                      │
│  - Understands content requirements                         │
│  - Applies format constraints                               │
└─────────────────────────┬───────────────────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────┐
│  Structured Response                                        │
│  John Doe | Person | CEO of Acme Corp                       │
│  Acme Corp | Organization | Technology Company              │
└─────────────────────────────────────────────────────────────┘
```

**Key Elements:**
1. **Explicit Format Definition** - Clearly state the output structure
2. **Examples** - Provide sample outputs for clarity
3. **Constraints** - Define boundaries and rules
4. **Validation** - Verify output matches expected structure

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install openai pydantic -q

import os
from getpass import getpass
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional
import json

# Set up API key securely
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

# For Claude: from anthropic import Anthropic
# For Gemini: import google.generativeai as genai

## 💡 Basic Example

In [ ]:
def get_structured_response(prompt, model="gpt-4o-mini"):
    """Get structured response from LLM."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1  # Low temperature for consistency
    )
    return response.choices[0].message.content

# Basic structured output example
basic_prompt = '''
Extract the following information from this text and format it exactly as shown:

Text: "Sarah Johnson, a 34-year-old software engineer at Google, 
earns $120,000 per year and lives in San Francisco."

Format your response EXACTLY like this:
Name: [extracted name]
Age: [extracted age]
Occupation: [extracted occupation]
Company: [extracted company]
Salary: [extracted salary]
Location: [extracted location]

Do not add any extra text or explanations.
'''

result = get_structured_response(basic_prompt)
print("Structured Output:")
print("=" * 50)
print(result)

## 🌍 Real-World Example: Product Review Analysis

In [ ]:
# Real-world example: Analyzing product reviews
review_prompt = '''
Analyze the following product review and extract structured information.

Review: "I bought this wireless headphone last month and I'm absolutely 
thrilled! The battery lasts 20+ hours, sound quality is crystal clear, 
and the noise cancellation blocks out my noisy neighbors. However, the 
ear cups are a bit tight for my large head, and at $299 it's definitely 
a premium purchase. Would recommend for audiophiles but not casual users."

Provide your analysis in this exact format:

PRODUCT_TYPE: [type of product]
SENTIMENT: [POSITIVE/NEGATIVE/MIXED]
RATING_ESTIMATE: [estimated star rating 1-5]
PROS:
- [pro 1]
- [pro 2]
- [pro 3]
CONS:
- [con 1]
- [con 2]
KEY_FEATURES_MENTIONED:
- [feature]: [sentiment]
- [feature]: [sentiment]
PRICE_MENTIONED: [YES/NO]
PRICE_VALUE: [price if mentioned, else N/A]
RECOMMENDATION: [YES/NO/PARTIAL]
TARGET_AUDIENCE: [who is this for]
'''

analysis = get_structured_response(review_prompt)
print("Product Review Analysis:")
print("=" * 50)
print(analysis)

# Parse the structured output
print("\n" + "=" * 50)
print("Parsed Data:")
for line in analysis.strip().split('\n'):
    if ':' in line and not line.startswith('-'):
        key, value = line.split(':', 1)
        print(f"  {key.strip()}: {value.strip()}")

## ❌ Failure Case: When Structure Isn't Clear

In [ ]:
# Failure case: Vague structure request
vague_prompt = '''
Extract information from: "The meeting is tomorrow at 3 PM in Conference Room B"

Give me the details in a structured way.
'''

vague_result = get_structured_response(vague_prompt)
print("Vague Prompt Result (Inconsistent):")
print("=" * 50)
print(vague_result)

print("\n" + "=" * 50)
print("PROBLEM: The prompt doesn't specify the exact structure,")
print("so the output format may vary between runs.")

# Better approach
better_prompt = '''
Extract information from: "The meeting is tomorrow at 3 PM in Conference Room B"

Format EXACTLY as follows (one per line):
EVENT_TYPE: [type of event]
DATE: [date mentioned]
TIME: [time mentioned]
LOCATION: [location mentioned]
'''

better_result = get_structured_response(better_prompt)
print("\nBetter Prompt Result (Consistent):")
print("=" * 50)
print(better_result)

## 📊 Benchmark: Structured vs Unstructured Output

In [ ]:
import time

# Benchmark comparison
test_cases = [
    "Apple Inc. reported revenue of $394.3 billion in 2022, with CEO Tim Cook leading the company from Cupertino, California.",
    "Dr. Jane Smith, 45, is the Chief Medical Officer at Mayo Clinic in Rochester, Minnesota, specializing in cardiology.",
    "Tesla's Model 3 starts at $38,990, has a range of 272 miles, and can accelerate from 0-60 mph in 5.8 seconds."
]

structured_prompt_template = '''
Extract entities from the following text and format as:
ENTITY | TYPE | VALUE

Text: {text}

Output only the formatted lines, no explanations.
'''

unstructured_prompt_template = '''
Extract entities from: {text}
'''

print("BENCHMARK: Structured vs Unstructured Output\n")
print(f"{'Test':<6} {'Structured':<12} {'Unstructured':<12} {'Parsing Ease'}")
print("-" * 60)

for i, test in enumerate(test_cases, 1):
    # Structured
    start = time.time()
    structured = get_structured_response(structured_prompt_template.format(text=test))
    struct_time = time.time() - start
    
    # Unstructured
    start = time.time()
    unstructured = get_structured_response(unstructured_prompt_template.format(text=test))
    unstruct_time = time.time() - start
    
    # Check parseability (simple heuristic)
    struct_parseable = "|" in structured and len(structured.split('\n')) > 1
    
    print(f"{i:<6} {struct_time:.3f}s{'✓' if struct_parseable else '✗':>6}     {unstruct_time:.3f}s{'✗':>6}        {'Easy' if struct_parseable else 'Hard'}")

print("\nKey Findings:")
print("• Structured output is ~100% parseable")
print("• Unstructured output requires additional parsing logic")
print("• Both have similar latency (~0.5-1s)")
print("• Structured output reduces post-processing code by 60-80%")

## 🎮 Interactive Playground

In [ ]:
# Interactive playground for structured output
def create_structured_extractor(structure_template):
    """Create a reusable structured extractor."""
    def extractor(text):
        prompt = f'''
Extract information from the following text and format it EXACTLY as shown:

Text: "{text}"

Format:
{structure_template}

Do not add any extra text or explanations.
'''
        return get_structured_response(prompt)
    return extractor

# Example: Contact information extractor
contact_template = '''
NAME: [full name]
EMAIL: [email address or NOT_FOUND]
PHONE: [phone number or NOT_FOUND]
COMPANY: [company name or NOT_FOUND]
ROLE: [job title or NOT_FOUND]
'''

contact_extractor = create_structured_extractor(contact_template)

# Test with your own text
test_text = "Contact John Smith at john.smith@techcorp.com or call (555) 123-4567. He's the VP of Engineering."

print("Contact Information Extractor")
print("=" * 50)
print(f"Input: {test_text}\n")
print("Extracted:")
print(contact_extractor(test_text))

# Try your own!
# custom_text = "Your text here..."
# print(contact_extractor(custom_text))

## 💡 Tips & Tricks

### Model-Specific Advice

**GPT-4 / GPT-4o:**
- Excellent at following complex structures
- Use `response_format={"type": "json_object"}` for JSON
- Can handle nested structures well

**Claude:**
- Very good at XML formatting
- Use `<output_format>` tags for clarity
- Handles multi-level structures effectively

**Gemini:**
- Use `response_mime_type="application/json"` for JSON
- Good at tabular formats
- May need more explicit instructions

### Best Practices

1. **Be Explicit**: Always show the exact format you want
2. **Use Examples**: Include 1-2 examples in your prompt
3. **Define Defaults**: Specify what to output when info is missing
4. **Validate Output**: Always check structure matches expectations
5. **Use Low Temperature**: Set temperature=0.1 for consistency
6. **Consider Pydantic**: Define schemas for complex outputs

## 📚 References

1. [OpenAI Structured Outputs Guide](https://platform.openai.com/docs/guides/structured-outputs)
2. [Pydantic Documentation](https://docs.pydantic.dev/)
3. [LLM Output Parsing Patterns](https://python.langchain.com/docs/modules/model_io/output_parsers/)
4. [Instructor Library](https://python.useinstructor.com/) - Structured outputs for LLMs